In [27]:
# !pip install 'pydantic[email]'

## What is the need for pydantic?
    - API development ( FastAPI uses pydantic heavely )
    - LLM interaction - structured data
    - Agentic solution

In [3]:
# we want to create user which has  name , age and email

def create_user(name, age, email):
    return {"name": name, "age": age, "email": email }

# create_user(name = "Ramesh", age = 20, email = "ramesh@gmail.com")
create_user(name = "Ramesh", age = "twenty", email = "ramesh@gmail.com")

{'name': 'Ramesh', 'age': 'apple', 'email': 'ramesh@gmail.com'}

In [4]:
def create_user(name : str, age: int, email: str): # type hinting
    return {"name": name, "age": age, "email": email }

# create_user(name = "Ramesh", age = 20, email = "ramesh@gmail.com")
create_user(name = "Ramesh", age = "twenty", email = "ramesh@gmail.com")

{'name': 'Ramesh', 'age': 'twenty', 'email': 'ramesh@gmail.com'}

In [8]:
def create_user(name : str, age: int, email: str): # type hinting

    if not isinstance(name, str):
        raise ValueError("Name of incorrect type!")
        
    if not isinstance(age, int):
        raise ValueError("Age is of incorrect type!")

    
        
    return {"name": name, "age": age, "email": email }

create_user(name = "Ramesh", age = "20", email = "ramesh@gmail.com")

ValueError: Age is of incorrect type!

In [17]:

class User:
    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email

user = User(name = "Ramesh", age = 20, email = "ramesh@gmail.com")
user.name, user.age, user.email

('Ramesh', 20, 'ramesh@gmail.com')

In [11]:
from dataclasses import dataclass

In [15]:
@dataclass
class User:
    name: str
    age: int
    email: str

user = User(name = "Ramesh", age = 20, email = "ramesh@gmail.com")
user.name, user.age, user.email

('Ramesh', 20, 'ramesh@gmail.com')

## Data Validation using Pydantic

In [9]:
from pydantic import BaseModel

In [21]:
class User(BaseModel):
    name: str
    age: int
    email: str

# user = User(name = "Ramesh", age = "20", email = "ramesh@gmail.com")

User(name = "Ramesh", age = "twenty", email = "ramesh@gmail.com")
user.name, user.age, user.email

ValidationError: 1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

    When you create a Pydantic model instance:

        Parsing: Pydantic attempts to convert input values to the declared types. If you pass "25" where int is expected, Pydantic converts it to 25 (unless strict mode is on).
        Validation: Each field is checked against its constraints (min_length, ge, le, etc.).
        Error collection: If multiple fields fail, Pydantic collects all errors and raises them together, not one at a time.
        Object creation: If everything passes, you get a Python object with attribute access (user.name) and type-safe methods.

            This happens automatically every time you create a model instance. The validation is exhaustive and the errors are structured.

### Add constraints on each field

In [31]:
from pydantic import EmailStr, Field

class User(BaseModel):
    name: str = Field(..., min_length = 1, max_length = 10 )
    age: int = Field(..., ge=0, le  = 100)
    email: EmailStr # will validate that email is a valid email

# user = User(name = "Ramesh", age = "20", email = "ramesh@gmail.com")

User(name = "Ramesh Kumar", age = "twenty", email = "rameshgmail.com")
user.name, user.age, user.email

ValidationError: 3 validation errors for User
name
  String should have at most 10 characters [type=string_too_long, input_value='Ramesh Kumar', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_long
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='rameshgmail.com', input_type=str]

In [33]:
class Product(BaseModel):
    name: str
    price: float
    in_stock: bool

# Create an instance
product = Product(name="Cutting Chai", price=15.0, in_stock=True)

# Access the fields
print(product.name)        # "Cutting Chai"
print(product.price)       # 15.0
print(product.in_stock)    # True

Cutting Chai
15.0
True


In [34]:
## serialise a object from pydantic to a dictionary or to json

product.model_dump() # # convert the instance to a dictionary

{'name': 'Cutting Chai', 'price': 15.0, 'in_stock': True}

In [35]:
product.model_dump_json()

'{"name":"Cutting Chai","price":15.0,"in_stock":true}'

In [36]:
product.model_json_schema()  # used for creating openAPI schemas

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'price': {'title': 'Price', 'type': 'number'},
  'in_stock': {'title': 'In Stock', 'type': 'boolean'}},
 'required': ['name', 'price', 'in_stock'],
 'title': 'Product',
 'type': 'object'}

In [39]:
## Invalid data examples

# product = Product(name="Samosa", price=15.0) # one field is missing

# product = Product(name="Samosa", price="not a number", in_stock=True)

product = Product(name=None, price="abc", in_stock="maybe")

ValidationError: 3 validation errors for Product
name
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type
price
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing
in_stock
  Input should be a valid boolean, unable to interpret input [type=bool_parsing, input_value='maybe', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/bool_parsing

In [40]:
product = Product(name="Chai", price="15", in_stock=True)
print(type(product.price))   # <class 'float'>
print(product.price)         # 15.0

<class 'float'>
15.0


### Different types in Fields

In [ ]:
from pydantic import BaseModel

class Example(BaseModel):
    # Text
    name: str           # Any string
    
    # Numbers
    age: int            # Integer
    price: float        # Floating-point number
    
    # Boolean
    is_active: bool     # True or False
    
    # Bytes
    data: bytes         # Binary data (file contents, etc.)

In [ ]:
from datetime import datetime, date, time, timedelta
from pydantic import BaseModel

class Event(BaseModel):
    scheduled_at: datetime   # Date and time: 2024-09-15T14:30:00
    event_date: date         # Date only: 2024-09-15
    start_time: time         # Time only: 14:30:00
    duration: timedelta      # Duration: P2H30M (2 hours 30 minutes)

# Pydantic accepts multiple formats
event = Event(
    scheduled_at="2024-09-15T14:30:00",    # ISO 8601 string
    event_date="2024-09-15",                # ISO date string
    start_time="14:30:00",                  # ISO time string
    duration=9000                           # Seconds (9000s = 2h30m)
)

In [41]:
from uuid import UUID, uuid4
from pydantic import BaseModel

class Transaction(BaseModel):
    id: UUID
    amount: float

# Pass a UUID string or a UUID object
t = Transaction(id="550e8400-e29b-41d4-a716-446655440000", amount=100.0)
t = Transaction(id=uuid4(), amount=100.0)

In [42]:
from enum import Enum
from pydantic import BaseModel

class OrderStatus(str, Enum):
    PENDING = "pending"
    PREPARING = "preparing"
    READY = "ready"
    DELIVERED = "delivered"

class Order(BaseModel):
    id: int
    status: OrderStatus

# Valid: accepts the enum value or its string value
order = Order(id=1, status="pending")          # Works
order = Order(id=1, status=OrderStatus.READY)  # Works

# Invalid: any other value rejected
order = Order(id=1, status="cooking")
# ValidationError: Input should be 'pending', 'preparing', 'ready' or 'delivered'

ValidationError: 1 validation error for Order
status
  Input should be 'pending', 'preparing', 'ready' or 'delivered' [type=enum, input_value='cooking', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/enum

### Optional Fields and Default values

In [48]:
from pydantic import BaseModel
from typing import Optional


class User(BaseModel):
    username: str
    email: str
    # bio: str | None   ## bio can be str or None but it is mandatory
    bio: str | None = None  ## here there is a default value --> bio can be skipped but it should be str or None if present
    # bio : Optional[str] = None  # alternate way of writing


user = User(username="rahul", email="rahul@example.com") # bio is not provided
print(user.bio)  # None




None


    Just adding Optional[str] without a default does NOT make the field optional. It only widens the type to allow None. The field is still required (but can be None).

In [53]:
## Default Values

class Product(BaseModel):
    name : str
    price: float
    is_available: bool = True # True is the default value
    stock_amount: int = 0
    tags : list[str] = [] ## default value is a empty list
    metadata: dict = {} # default value is empty dictionary

    
product = Product(name="Samosa", price=15.0)
product = Product(name="Samosa", price=15.0, tags = ["indian", "snack"], metadata= {'calories': 150})
product.model_dump()

{'name': 'Samosa',
 'price': 15.0,
 'is_available': True,
 'stock_amount': 0,
 'tags': ['indian', 'snack'],
 'metadata': {'calories': 150}}

In [55]:
# DANGEROUS in regular Python
def add_item(item, tags=[]):  # bad idea to have mutable type as default
    tags.append(item)
    return tags
print(add_item("a"))
print(add_item("b"))

['a']
['a', 'b']


In [ ]:
fr

### Default factory : for dynamic defaults

```default_factory``` accepts a callable (a function with no arguments) that returns the default value.

In [57]:
from datetime import datetime
str(datetime.now())

'2026-04-21 20:16:42.531071'

In [62]:
from uuid import uuid4
uuid4()

UUID('52e0fe70-80ed-4a4d-93fe-7b9020c715e8')

In [64]:
class Event(BaseModel):
    id: str = Field(default_factory= lambda : (uuid4())  )
    created_at: datetime = Field(default_factory=datetime.now)
    name : str

event1 = Event( name = "Ramesh")

event2 = Event( name = "Mahesh")

print(event1)
print(event2)

id=UUID('7b80bde7-095e-4535-840f-fa0c1b77fe66') created_at=datetime.datetime(2026, 4, 21, 20, 21, 19, 899518) name='Ramesh'
id=UUID('908d1d49-2ae4-4a85-b9a3-424220d1cd69') created_at=datetime.datetime(2026, 4, 21, 20, 21, 19, 899618) name='Mahesh'


### Field Function
    - str field
    - numeric field
    - collection field ( list, set, tuple)
    - we can also add metadata in field

In [82]:
class User(BaseModel):
    username: str = Field(..., 
                          min_length = 3, 
                          max_length=10,
                          pattern=r"^[A-Za-z0-9_]+$")
    bio: str = Field(default = "", max_len = 500)

user1 = User(username= "Rames", bio = "I am a happy man. ")
user1.model_dump()

/var/folders/st/b0hjmx2s42dbbvjsct0_5c300000gn/T/ipykernel_16402/1469365464.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'max_len'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  bio: str = Field(default = "", max_len = 500)


{'username': 'Rames', 'bio': 'I am a happy man. '}

In [85]:
class Product(BaseModel):
    price: float = Field(
        ...,
        gt=0,                  # Greater than 0
        le=10000               # Less than or equal to 10000
    )
    rating: int = Field(
        default=0,
        ge=0,                   # Greater than or equal to 0
        le=5                    # Less than or equal to 5
    )
    discount_percent: float = Field(
        default=0,
        ge=0,
        lt=100, # Less than 100 (not equal)
        # multiple_of=5
    )

Product(price = 100,discount_percent= 11)

Product(price=100.0, rating=0, discount_percent=11.0)

In [90]:
class Order(BaseModel):
    items : list[str] = Field(..., min_length=1, max_length=10 )
    tags: set[str] = Field(default = set(), max_length = 5)

order1 = Order(items = ["samosa", "coke", "biryani"])
order1.model_dump()

{'items': ['samosa', 'coke', 'biryani'], 'tags': set()}

    - Adding metadata to the fields

In [94]:
class Product(BaseModel):
    name: str = Field(
        ...,
        description="The product name, displayed to customers",
        examples=["Cutting Chai", "Samosa", "Masala Chai"]
    )
    price: float = Field(
        ...,
        description="Price in INR",
        gt=0,
        examples=[15.0, 20.0, 30.0]
    )
    sku: str = Field(
        ...,
        alias="productCode",    # Accept "productCode" in input
        description="Stock-keeping unit identifier"
    )

product = Product(name = "samosa", price = 10, productCode = "food")
product.model_json_schema()

{'properties': {'name': {'description': 'The product name, displayed to customers',
   'examples': ['Cutting Chai', 'Samosa', 'Masala Chai'],
   'title': 'Name',
   'type': 'string'},
  'price': {'description': 'Price in INR',
   'examples': [15.0, 20.0, 30.0],
   'exclusiveMinimum': 0,
   'title': 'Price',
   'type': 'number'},
  'productCode': {'description': 'Stock-keeping unit identifier',
   'title': 'Productcode',
   'type': 'string'}},
 'required': ['name', 'price', 'productCode'],
 'title': 'Product',
 'type': 'object'}

### Specialised types ( for email, url, IP address)

In [101]:
from pydantic import BaseModel, EmailStr, HttpUrl, IPvAnyAddress

class Contact(BaseModel):
    email: EmailStr              # Validates email format (requires email-validator package)
    website: HttpUrl             # Validates URL format
    server_ip: IPvAnyAddress     # IPv4 or IPv6 address

contact = Contact(
    email="rahul@example.com",
    website="https://rahul.dev",
    server_ip="192.168.1.1"
)
print(contact)

# Invalid email rejected
contact = Contact(email="email@email.com", website="http://codeverra.com", server_ip="13")
print(contact)
# ValidationError: email: value is not a valid email address

email='rahul@example.com' website=HttpUrl('https://rahul.dev/') server_ip=IPv4Address('192.168.1.1')


ValidationError: 1 validation error for Contact
server_ip
  value is not a valid IPv4 or IPv6 address [type=ip_any_address, input_value='13', input_type=str]

In [ ]:
from pydantic import BaseModel, PositiveInt, NegativeInt, NonNegativeInt, NonNegativeFloat

class Measurements(BaseModel):
    quantity: PositiveInt         # Must be > 0
    temperature_c: NegativeInt    # Must be < 0
    count: NonNegativeInt          # Must be >= 0
    weight_kg: NonNegativeFloat    # Must be >= 0

In [102]:
from pathlib import Path
from pydantic import BaseModel, FilePath, DirectoryPath

class Config(BaseModel):
    log_file: FilePath            # Path must exist and be a file
    data_dir: DirectoryPath       # Path must exist and be a directory
    output_path: Path              # Any path (no validation of existence)

### Nested Models

In [113]:
# Basic Nesting

class Address(BaseModel):
    door_number: int
    building_number: int
    street: str
    district: Optional[str] = None
    pincode: int 


class User(BaseModel):
    name: str
    email: str
    address: Address ## nested model

# Create with nested dict
user = User(
    name="Rahul Sharma",
    email="rahul@example.com",
    address={
        "door_number": 30,
        "building_number": 32,
        "street": "123 MG Road",
        "city": "Bangalore",
        "state": "Karnataka",
        "pincode": 500084
    }
)

user.model_dump_json()


'{"name":"Rahul Sharma","email":"rahul@example.com","address":{"door_number":30,"building_number":32,"street":"123 MG Road","district":null,"pincode":500084}}'

In [114]:
## Deep Nesting

class Address(BaseModel):
    street: str
    city: str
    state: str
    pincode: str

class Coordinates(BaseModel):
    latitude: float
    longitude: float
    

class Location(BaseModel):
    address: Address
    coordinates: Coordinates

class Restaurant(BaseModel):
    name : str
    cuisine: str
    location: Location



restaurant = Restaurant(
    name="Paradise Biryani",
    cuisine="Hyderabadi",
    location={
        "address": {
            "street": "Shop 5, Paradise Circle",
            "city": "Hyderabad",
            "state": "Telangana",
            "pincode": "500003"
        },
        "coordinates": {
            "latitude": 17.4342,
            "longitude": 78.4489
        }
    },
    rating=4.5
)


restaurant.model_dump_json()

'{"name":"Paradise Biryani","cuisine":"Hyderabadi","location":{"address":{"street":"Shop 5, Paradise Circle","city":"Hyderabad","state":"Telangana","pincode":"500003"},"coordinates":{"latitude":17.4342,"longitude":78.4489}}}'

### Using list, dict and collections in pydantic

In [122]:
class Comment(BaseModel):
    author : str ## author should be of type User. here we are using string
    text: str

class Post(BaseModel):
    title: str
    tags: list[str]
    comments: list[Comment]


post = Post(
    title="My First Post",
    tags=["python", "tutorial", "beginners"],
    comments = [
        Comment(author = "lalit", text = "wow wonderful!!"), 
        {"author": "ram", "text": "great post. " }, 
               {"author": "shyam", "text": "terrible post. Delete your account!" }],
    view_counts=100 # ignored by default
)

post.model_dump()

{'title': 'My First Post',
 'tags': ['python', 'tutorial', 'beginners'],
 'comments': [{'author': 'lalit', 'text': 'wow wonderful!!'},
  {'author': 'ram', 'text': 'great post. '},
  {'author': 'shyam', 'text': 'terrible post. Delete your account!'}]}

In [126]:
type(post.comments[0])

__main__.Comment

In [ ]:
# Dictionaries as types in field

class Product(BaseModel):
    name: str
    attributes: dict[str, str]           # Keys and values are strings
    prices_by_size: dict[str, float]    # Keys are strings, values are floats

product = Product(
    name="T-Shirt",
    attributes={"color": "blue", "material": "cotton"},
    prices_by_size={"S": 499, "M": 599, "L": 699}
)

In [127]:
# set and tuples as type in fields


class Config(BaseModel):
    allowed_ips: set[str]                  # Unique values, order does not matter
    dimensions: tuple[int, int, int]        # Fixed-length tuple (width, height, depth)
    point: tuple[float, float]              # 2D point

config = Config(
    allowed_ips={"192.168.1.1", "192.168.1.2"},
    dimensions=(100, 200, 50),
    point=(17.4342, 78.4489)
)

In [132]:
## nested collections

class Classroom(BaseModel):
    students_by_section: dict[str, list[str]]
    # {"A": ["Rahul", "Priya"], "B": ["Amit", "Sneha"]}

class7 = Classroom(students_by_section =  {"A": ["Ram", "Shyam", "Mohan"],
                    "B": ["Amit", "Sneha"]})
print(class7)




class Matrix(BaseModel):
    values: list[list[float]]
    # [[1.0, 2.0], [3.0, 4.0]]

students_by_section={'A': ['Ram', 'Shyam', 'Mohan'], 'B': ['Amit', 'Sneha']}


In [135]:
from datetime import date

class Student(BaseModel):
    name : str
    dob: Optional[date] = None


class Classroom(BaseModel):
    students_by_section: dict[str, list[Student]]
    # {"A": ["Rahul", "Priya"], "B": ["Amit", "Sneha"]}

class7 = Classroom(students_by_section =  {"A": [Student(name = "Ram"), Student(name = "Shyam"),],
                    "B": [Student(name = "Amit")]})
print(class7)

students_by_section={'A': [Student(name='Ram', dob=None), Student(name='Shyam', dob=None)], 'B': [Student(name='Amit', dob=None)]}


### Custom Validators

    - field_validator
    - model_validator

- @field_validator

In [138]:
from pydantic import field_validator, model_validator

In [147]:
# Going to apply some rule on a gven field

class User(BaseModel):
    username: str
    email: str

    # @field_validator("username")
    # @classmethod
    # def process_username(cls, v: str):
    #     return v.title().strip() ## convert to lowercase and strip any leading or trailing space

    # @field_validator("email")
    # @classmethod
    # def process_email(cls, v: str):
    #     return v.lower().strip() ## convert to lowercase and strip any leading or trailing space

    @field_validator("username", "email")
    @classmethod
    def process_field(cls, v: str):
        return v.lower().strip() ## convert to lowercase and strip any leading or trailing space


user  = User(username = "LALIT", email = "LALIT@codeverra.com")
user.model_dump()

{'username': 'lalit', 'email': 'lalit@codeverra.com'}

    - Two important modes - Before, After (default)
    

In [146]:
class Product(BaseModel):
    price: float

    @field_validator("price", mode="before")
    @classmethod
    def parse_rupee_string(cls, v):
        """Accept prices as strings like 'Rs 15' or '15 INR'."""
        if isinstance(v, str):
            # Remove non-numeric characters
            cleaned = "".join(c for c in v if c.isdigit() or c == ".")
            return float(cleaned)
        return v

product = Product(price="Rs 15.50")
print(product.price)  # 15.5

product = Product(price="15 INR")
print(product.price)  # 15.0

product = Product(price=20)
print(product.price)  # 20.0

ValidationError: 1 validation error for Product
price
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='Rs 15.50', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing

- @model_validator
    - A class method that will validate the entire model. 

In [156]:
class DateRange(BaseModel):
    start_date: str
    end_date: str

    @model_validator(mode = "after")
    # @classmethod
    def validate_daterange(self) :
        if self.start_date > self.end_date:
            raise ValueError("Start date must be on or before end data")
        return self

date_range = DateRange(start_date  = "2026-10-06", end_date = "2026-10-02")
date_range.model_dump_json()

ValidationError: 1 validation error for DateRange
  Value error, Start date must be on or before end data [type=value_error, input_value={'start_date': '2026-10-0...end_date': '2026-10-02'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [159]:
class Password(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode = "after")
    def model_validator(self):
        if self.password != self.confirm_password:
            raise ValueError("passwords dont match")
        return self

password = Password(password = "admin123", confirm_password = "admin1234")

ValidationError: 1 validation error for Password
  Value error, passwords dont match [type=value_error, input_value={'password': 'admin123', ..._password': 'admin1234'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [ ]:
## Serialization
    - model_dump
    - model_dump_json
    - exclude_unset 
    - exclude_none

In [161]:
class User(BaseModel):
    id: int
    username: str
    bio: str | None = None
    is_active: bool = True

# Only username and id are set; others have defaults
user = User(id=1, username="rahul")

print(user.model_dump())
# {"id": 1, "username": "rahul", "bio": None, "is_active": True}

print(user.model_dump(exclude_unset=True))
# {"id": 1, "username": "rahul"}
# Fields with defaults are excluded

{'id': 1, 'username': 'rahul', 'bio': None, 'is_active': True}
{'id': 1, 'username': 'rahul'}


### Generic Models

### Custom Settings in Pydantic - Model Configuration

In [ ]:
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    model_config = ConfigDict(
        # Allow creating from object attributes (not just dicts)
        from_attributes=True,
        
        # Validate when values are assigned to fields
        validate_assignment=True,
        
        # Make instances immutable (frozen)
        frozen=True,
        
        # Control how extra fields in input are handled
        extra="forbid",  # or "allow" or "ignore"
        
        # Accept enum values instead of enum instances
        use_enum_values=True,
        
        # Strip whitespace from strings
        str_strip_whitespace=True,
        
        # Convert strings to lowercase
        str_to_lower=False
    )
    
    username: str
    email: str

In [172]:
from pydantic import BaseModel, ConfigDict
class User(BaseModel):

    model_config = ConfigDict(
        validate_assignment=True, 
        frozen = True, 
        extra="forbid"
        
    )
    name: str
    age : float


user = User(name = "Lalit", age = "30", email = "lalit@gmail.com") # instatiation / creating the object
user.model_dump()

ValidationError: 1 validation error for User
email
  Extra inputs are not permitted [type=extra_forbidden, input_value='lalit@gmail.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden

In [170]:
user.age = "apple"  # while assignment the validation rules dont apply(by default)

ValidationError: 1 validation error for User
age
  Instance is frozen [type=frozen_instance, input_value='apple', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/frozen_instance

In [166]:
user.model_dump()

/opt/miniconda3/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `float` - serialized value may not be as expected [field_name='age', input_value='apple', input_type=str])
  return self.__pydantic_serializer__.to_python(


{'name': 'Lalit', 'age': 'apple'}

## Generic Model

In [ ]:
# You write this once...
class UserResponse(BaseModel):
    success: bool
    data: User | None
    error: str | None

# ...and then copy-paste for every new type
class PostResponse(BaseModel):
    success: bool
    data: Post | None     # Only this line changes
    error: str | None

class CommentResponse(BaseModel):
    success: bool
    data: Comment | None  # Only this line changes
    error: str | None

# 10 resources = 10 nearly identical classes. Tedious.